# Unified Conflict-Evaluation Pipeline

Four-layer evaluation, one entry point: `evaluate(model, tokenizer, items, ...)`.

| Layer | Runs on | Purpose | Output |
|---|---|---|---|
| 0 — genuine-conflict filter | no-context | is this item eligible? | pass/fail |
| 1 — log-prob scoring | with-context | primary measurement | Δ, CTX/PAR/AMBIG |
| 2 — LLM judge | AMBIG items only | resolve the ambiguous middle | CTX/PAR/OTHER |
| 3 — paper / ordered | same generation as L2 | agreement diagnostics only | CTX/PAR/OTHER |

**final_label** = Layer 1 verdict if `|Δ| ≥ τ`, else Layer 2 verdict. This is what `R_ctx`
is computed from. `paper` and `ordered` are never used to compute the headline rate —
they exist to catch cases where Layers 1+2 might themselves be wrong.

Validation is staged: offline self-tests (no GPU) → manual preflight on adversarial
items (small model call) → Phase-0 full run → agreement/timing tests — **before**
this is handed to the SFT loop as `evaluate_checkpoint(...)`.

In [78]:
import os, json, re,gc,time,hashlib,unicodedata,random
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM
import math

#Configurations:

MODELS = [
    {"id": "meta-llama/Llama-3.1-8B-Instruct",     "use_chat_template": True},
    {"id": "mistralai/Mistral-7B-Instruct-v0.1",   "use_chat_template": True},
    {"id": "Qwen/Qwen2-7B-Instruct",               "use_chat_template": True}
]


CONFLICT_JSON = "/content/drive/MyDrive/context-parametric-inversion-research/dataset/conflict_eval_unified.json"
OUT_DIR       = "/content/drive/MyDrive/context-parametric-inversion-research/phase0_results"

os.makedirs(OUT_DIR, exist_ok=True)

PREFILL        = "The answer is:" #assistant-turn prefill for log-prob scoring (no-trailing spaces)
MAX_NEW_TOKENS = 24
TAU            = 1   # log-prob confidence gate (nats); pre-registered, do not tune per-checkpoint
K_DISTRACTORS=5 # for the geniuine-conflict filter

DROP_IDS = {
    "cap_0020","cap_0027","cap_0041","cap_0121","cap_0161"
}

### Authentication for Llama : As Llama is gated model requires hugging face access

In [76]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("colab_token"))
print("HuggingFace login successful.")

HuggingFace login successful.


### Model Loading

`load_model` refuses silent 4-bit loading , it invalidates our cross-checkpoint log-prob comparatibility, so it must be opted into explicitly

In [45]:
def load_model(model_id, precision="bf16", allow_4bit=False):
    assert precision in ("bf16", "4bit")
    if precision == "4bit":
        assert allow_4bit, (
            "4-bit loading degrades log-prob scoring and breaks cross-checkpoint "
            "comparability. Set allow_4bit=True explicitly if you accept this.")

    print(f"Loading {model_id}  precision={precision}")
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    kw = dict(torch_dtype=torch.bfloat16, device_map="auto")
    if precision == "4bit":
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    if hasattr(model, "get_memory_footprint"):
        print(f"  Memory footprint: {model.get_memory_footprint()/1e9:.2f} GB (precision={precision})")
    return model, tok

def free_model(model):
    del model; gc.collect(); torch.cuda.empty_cache()
    print("GPU memory released.")

## Data

Unified schema (matches both the 413-item `conflict_eval_unified.json` and the
20-item `PROBE_SAMPLES` from the earlier notebook):
`item_id, source, question, context, parametric_answer, counterfactual_answer`.

The loader uses the JSON if present, else falls back to `PROBE_SAMPLES` (your
existing 20-item stratified probe), so Phase 0 runs unchanged.

The loader loads the `conflict_eval_unified.json`, validates our schema and drops the items flagged in the pipeline spec (unusable parametric answers - multi facted answer and a consensus generation check was performed and all 3 models noted inconsistencies in the parametric answer without context) 5 items

In [46]:
REQUIRED = {"item_id","source","question","context","parametric_answer","counterfactual_answer"}


def load_items(path, drop_ids=DROP_IDS):
    data = json.load(open(path))
    items = data["items"] if isinstance(data, dict) and "items" in data else data
    for it in items:
        missing = REQUIRED - set(it)
        assert not missing, f"{it.get('item_id','?')} missing {missing}"
    kept = [it for it in items if it["item_id"] not in drop_ids]
    dropped = len(items) - len(kept)
    print(f"Loaded {len(items)} total | dropped {dropped} flagged item(s) | {len(kept)} usable")
    return kept

In [47]:
items = load_items(CONFLICT_JSON)
id_map = {it["item_id"]: it for it in items}
print("strata:", {s: sum(i["source"]==s for i in items) for s in sorted({i["source"] for i in items})})

Loaded 421 total | dropped 5 flagged item(s) | 416 usable
strata: {'country_capitals': 191, 'famous_biographies': 109, 'world_facts': 116}


### Prompts and Scoring Primitives

- `normalize` strips accents/case/whitespace so string comparisons are robust (Brasilia=brasilia).
- `contains/first_pos` use **word-boundary regex**, not substring search
- Generation prompts: Instruction to answer briefly, wrapped inside the chat template.
- Score prompts: same user turn, with `PPREFILL` ("The answer is ) appended as the **start of the assistant turn**, so candidate answers are scored in the true generation position.


In [48]:
STOP = {"the","a","an","of","in","on","at","and","or","is","was","to","for"}

def normalize(t):
  s = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
  return re.sub(r"\s+", " ", s).strip().lower()

def sig_words(ans):
    return [w for w in normalize(ans).split() if len(w) > 3 and w not in STOP]

def discr_words(ans, other):
    o = set(normalize(other).split())
    dw = [w for w in sig_words(ans) if w not in o]
    return dw if dw else sig_words(ans)

def _bounded(w):
    return r"(?<!\w)" + re.escape(w) + r"(?!\w)"

def contains(resp, ans, words):
    r = normalize(resp)
    if not words:
        return bool(re.search(_bounded(normalize(ans)), r))
    return any(re.search(_bounded(w), r) for w in words)

def first_pos(resp, ans, words):
    r = normalize(resp)
    positions = []
    m = re.search(_bounded(normalize(ans)), r)
    if m: positions.append(m.start())
    for w in words:
        m = re.search(_bounded(w), r)
        if m: positions.append(m.start())
    return min(positions, default=10**9)


def _user_text(question, context):
    if context is None:
        return ("Answer the question in as few words as possible. "
                "Do not explain. Do not add notes.\n\nQuestion: " + question)
    return ("Read the context and answer the question in as few words as possible. "
            "Do not explain. Do not add notes.\n\nContext: " + context +
            "\n\nQuestion: " + question)


In [49]:
@torch.no_grad()
def generate(model, tok, prompt_ids, max_new_tokens=MAX_NEW_TOKENS):
    ids = prompt_ids.unsqueeze(0).to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()


In [50]:
@torch.no_grad()
def score_answer_logprob(model, tok, prompt_ids, answer_text):
    """
    log P(answer | prompt), teacher-forced, plus token count for length normalisation.

        log P(a|prompt) = sum_i log P(t_i | prompt, t_<i})

    One forward pass on [prompt_ids ++ answer_ids]; logits at position i predict
    token i+1, so the answer's own logits live at positions [n-1 .. n+len(answer)-2]
    where n = len(prompt_ids) — proven correct with synthetic logits in the Cell 15
    self-test, extended below to also cover the EOS-strip branch.

    Hardening:
    - Space handling centralised here, idempotent (strip, then prepend exactly one
      space) — safe regardless of whether the caller pre-adds a leading space, so
      filter_logprob (your Cell 8) needs no changes.
    - Trailing-EOS guard on the prompt: if present, strip it BEFORE computing n,
      so a stray EOS in the prompt can't shift the indexing.
    - Empty-answer guard: returns (0.0, 0) explicitly rather than relying on the
      slice arithmetic to degrade gracefully by accident.
    - Deliberately NOT masking by pad_token_id — in this pipeline pad_token_id ==
      eos_token_id (load_model() sets pad_token = eos_token), so that mask would
      zero out any legitimate EOS-valued token inside a real answer. It's also
      unreachable dead code here since `ans` is never padded to begin with.
    """
    answer_text = " " + answer_text.strip()
    ans = tok(answer_text, add_special_tokens=False, return_tensors="pt").input_ids[0]
    if ans.shape[0] == 0:
        return 0.0, 0

    if prompt_ids.numel() > 0 and prompt_ids[-1].item() == tok.eos_token_id:
        prompt_ids = prompt_ids[:-1]

    full = torch.cat([prompt_ids, ans]).unsqueeze(0).to(model.device)
    logits = model(full).logits[0]
    n = prompt_ids.shape[0]
    lp = torch.log_softmax(logits[n-1:-1, :].float(), dim=-1)
    tok_lp = lp[torch.arange(ans.shape[0]), ans.to(lp.device)]
    return tok_lp.sum().item(), ans.shape[0]

#### Tensor Coercion

In [51]:
def _to_tensor(ids):
    if isinstance(ids, str):
        raise TypeError("_to_tensor received a string — apply_chat_template returned "
                         "raw text, use the encode= path.")
    if isinstance(ids, torch.Tensor):
        return ids.squeeze().long()
    if hasattr(ids, "ids"):
        return torch.tensor(ids.ids, dtype=torch.long)
    return torch.tensor(list(ids), dtype=torch.long)

#### Applying chat template during generation

In [52]:
def _apply_template(tok, messages, add_generation_prompt=True):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    return tok(text, return_tensors="pt", add_special_tokens=False).input_ids[0]


### Prompt Builder

In [53]:
def build_gen_prompt_ids(tok, question, context, use_ct):
    u = _user_text(question, context)
    if use_ct and tok.chat_template:
        return _apply_template(tok, [{"role": "user", "content": u}])
    return tok(u + "\nAnswer:", return_tensors="pt", add_special_tokens=True).input_ids[0]

In [54]:
def build_score_prompt_ids(tok, question, context, use_ct, prefill=PREFILL):
    u = _user_text(question, context)
    if use_ct and tok.chat_template:
        base = _apply_template(tok, [{"role": "user", "content": u}])
    else:
        return tok(u + "\n" + prefill, return_tensors="pt", add_special_tokens=True).input_ids[0]
    pre = tok(prefill, add_special_tokens=False, return_tensors="pt").input_ids[0]
    return torch.cat([base, pre])

## Layer 0 — Genuine-conflict filter (distractor ranking)

$$\bar\ell(a) = \frac{1}{n}\sum_{i=1}^{n}\log P(t_i \mid \text{prompt}, t_{<i})$$

$$\text{PASS} \iff \bar\ell(\text{par}) = \max\big(\bar\ell(\text{par}), \bar\ell(\text{cf}), \bar\ell(d_1),\ldots,\bar\ell(d_K)\big)$$

$d_1,\dots,d_K$ ($K{=}5$) are answers from **other items in the same stratum**,
length-matched in tokens to `parametric_answer`. This is a recall test against a
distractor pool, not a binary par-vs-cf preference — the old version only checked
par > cf, which is why biography filter yield came back ~100% instead of the
expected 70–85%: preferring truth over one absurd counterfactual is trivial,
recalling the fact among plausible alternatives is not.

In [55]:
def _token_len(tok, s):
    return len(tok(" " + s.strip(), add_special_tokens=False).input_ids)

def build_distractor_pool(items):
    pool = {}
    for it in items:
        pool.setdefault(it["source"], []).append((it["item_id"], it["parametric_answer"]))
    return pool

### Distractor Function

In [56]:
def sample_distractors(tok, item, pool, k=K_DISTRACTORS, seed=0):
    src = item["source"]
    target_len = _token_len(tok, item["parametric_answer"])
    candidates = [(iid, ans) for iid, ans in pool[src] if iid != item["item_id"]]
    scored = sorted(candidates, key=lambda x: abs(_token_len(tok, x[1]) - target_len))
    band = scored[: max(k * 4, k)]
    rnd = random.Random(seed + (hash(item["item_id"]) % (2**16)))
    chosen = rnd.sample(band, k=min(k, len(band)))
    return [ans for _, ans in chosen]

### Filter Log probabibility Function

In [57]:
@torch.no_grad()
def filter_logprob(model, tok, item, pool, use_ct, k=K_DISTRACTORS, seed=0):
    q, par, cf = item["question"], item["parametric_answer"], item["counterfactual_answer"]
    pid = build_score_prompt_ids(tok, q, None, use_ct)

    def _score(a):
        lp, n = score_answer_logprob(model, tok, pid, " " + a.strip())
        return lp / max(n, 1)

    par_score = _score(par)
    cf_score  = _score(cf)
    distractor_scores = [_score(d) for d in sample_distractors(tok, item, pool, k=k, seed=seed)]
    return par_score == max([par_score, cf_score] + distractor_scores)

def filter_generation(model, tok, q, par, use_ct):
    """Secondary filter (generation-based), logged for comparison only — not used to gate R_ctx."""
    resp = generate(model, tok, build_gen_prompt_ids(tok, q, None, use_ct))
    return contains(resp, par, sig_words(par))

## Layer 1 — sequence log-prob scoring (primary measurement)

For candidate answer $a$ tokenised as $[t_1,\dots,t_n]$, teacher-forced on the
with-context score prompt:

$$\log P(a \mid \text{prompt}) = \sum_{i=1}^{n}\log P(t_i \mid \text{prompt}, t_{<i})$$

Length-normalised per-token log-probability:
$$\bar\ell(a) = \frac{1}{n}\log P(a\mid\text{prompt})$$

Logit gap (positive → model prefers context):
$$\Delta = \bar\ell(\text{counterfactual\_answer}) - \bar\ell(\text{parametric\_answer})$$

Confidence gate ($\tau = 1.0$ nat, pre-registered, fixed across all checkpoints):
$$\text{label} = \begin{cases}\text{CTX} & \Delta \ge \tau\\ \text{PAR} & \Delta \le -\tau\\ \text{AMBIG} & |\Delta| < \tau\end{cases}$$

`method_logprob` now takes a **prebuilt** `score_prompt_ids` rather than building
it internally. `evaluate()` builds it once, decodes it for the saved record, and
passes the same ids to both the scorer and (if escalated) reuses the same
generation for the judge — one canonical prompt per item, guaranteed identical
between what's scored and what's logged.

In [58]:
def method_logprob(model, tok, score_prompt_ids, par, cf, tau=TAU):
    slp, npar = score_answer_logprob(model, tok, score_prompt_ids, par)
    clp, ncf  = score_answer_logprob(model, tok, score_prompt_ids, cf)

    if npar == 0 or ncf == 0:
        # one of the two candidates tokenised to nothing — flag it rather than
        # let 0.0/max(n,1)=0.0 masquerade as "perfect confidence" and silently
        # produce a spurious CTX/PAR verdict
        return {"delta": 0.0, "label": "AMBIG", "lp_par_pt": 0.0, "lp_cf_pt": 0.0,
                "n_par": npar, "n_cf": ncf, "error": "empty_answer"}

    par_pt = slp / npar
    cf_pt  = clp / ncf
    d = cf_pt - par_pt
    label = "AMBIG" if abs(d) < tau else ("CTX" if d > 0 else "PAR")
    return {"delta": round(d, 4), "label": label,
            "lp_par_pt": round(par_pt, 3), "lp_cf_pt": round(cf_pt, 3),
            "n_par": npar, "n_cf": ncf}

## Layer 2 — LLM-as-judge (fires only on AMBIG)

$$\text{final\_label} = \begin{cases}\text{Layer 1 verdict (CTX/PAR)} & |\Delta|\ge\tau\\ \text{Layer 2 verdict (judge)} & |\Delta|<\tau\end{cases}$$

`gpt-4.1-nano`, temperature 0, JSON-only. Reads the *actual generation* (not a
scoring pass). Cached by `(item_id, checkpoint_id, sha256(response))` so re-running
a checkpoint's evaluation doesn't re-spend API budget. On persistent failure the
item is marked `OTHER` with `layer2_failed=True`, tracked separately from genuine
abstentions (spec §5.5).

**Addition beyond the original judge prompt:** the JSON schema now also asks for
`other_subtype` (`hedged` / `abstained` / `hallucinated` / `null`) — this was
already required by spec §6 ("Sub-categorisation of OTHER... Layer 2 provides
this breakdown directly") but the original prompt draft never actually asked for
it. Fixed here since it's free (a few extra tokens) and directly serves manual
verifiability — an `OTHER` row is much easier to eyeball with a reason attached.

Both the exact prompt sent and the raw (pre-parse) API response are stored on the
verdict, so every judge call is independently auditable from the saved JSON.

In [59]:
#%pip install -q openai>=1.0.0   # skip if already installed in this runtime

from openai import OpenAI

_api_key = userdata.get("OPENAI_API_KEY")
assert _api_key, "OPENAI_API_KEY not found in Colab secrets — add it via the key icon in the left sidebar."
_openai_client = OpenAI(api_key=_api_key)

JUDGE_MODEL = "gpt-4.1-nano"
_JUDGE_CACHE = {}   # (item_id, checkpoint_id, response_hash) -> verdict dict

def _judge_key(item_id, checkpoint_id, response_text):
    h = hashlib.sha256(response_text.encode("utf-8")).hexdigest()[:16]
    return (item_id, checkpoint_id, h)

def _judge_prompt(question, context, response, par, cf):
    return (
        "You are classifying a language model's answer to a factual question "
        "where the provided context contradicts common knowledge.\n\n"
        f"Question: {question}\nContext provided to the model: {context}\n"
        f"Model's response: {response}\n\n"
        "The two candidate answers are:\n"
        f"- PARAMETRIC (what the model knows from training): {par}\n"
        f"- CONTEXTUAL (what the context asserts): {cf}\n\n"
        'Respond with JSON only, matching this schema exactly:\n'
        '{"answer": "PAR" | "CTX" | "OTHER", '
        '"other_subtype": "hedged" | "abstained" | "hallucinated" | null}\n\n'
        "Rules:\n"
        f"- PAR if the response asserts {par} as the answer to the question\n"
        f"- CTX if the response asserts {cf} as the answer to the question\n"
        "- OTHER if the response abstains, hedges without committing, or gives a different answer entirely\n"
        "- other_subtype is null unless answer is OTHER. If answer is OTHER: "
        '"hedged" if both candidates appear without commitment, "abstained" if neither appears, '
        '"hallucinated" if a different, third answer is given\n'
        '- If the response is grammatically inverted (e.g. "capital of Kathmandu is Nepal") '
        "but the semantic answer is clear from intent, classify by intent"
    )

def judge_fn(response, question, context, par, cf, item_id="unknown",
             checkpoint_id="adhoc", max_retries=3):
    key = _judge_key(item_id, checkpoint_id, response)
    if key in _JUDGE_CACHE:
        return _JUDGE_CACHE[key]

    prompt = _judge_prompt(question, context, response, par, cf)
    last_err = None
    for attempt in range(max_retries):
        try:
            r = _openai_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=40,
                response_format={"type": "json_object"},
            )
            raw = r.choices[0].message.content
            parsed = json.loads(raw)
            label = parsed.get("answer")
            if label not in ("PAR", "CTX", "OTHER"):
                raise ValueError(f"unexpected judge label: {parsed!r}")
            verdict = {
                "label": label,
                "other_subtype": parsed.get("other_subtype"),
                "layer2_failed": False,
                "judge_prompt": prompt,
                "judge_raw_response": raw,
            }
            _JUDGE_CACHE[key] = verdict
            return verdict
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)

    verdict = {"label": "OTHER", "other_subtype": None, "layer2_failed": True,
               "error": str(last_err), "judge_prompt": prompt, "judge_raw_response": None}
    _JUDGE_CACHE[key] = verdict
    return verdict

## Layer 3 — cross-checks (diagnostics only, not the headline metric)

- `paper` → containment: `cf_acc`, `par_acc`. Both can be 1 (the paper's own
  convention — cf_acc *is* their context-following rate). Not a partition.
- `ordered` → CTX/PAR/OTHER by which candidate's discriminative words appear
  first; records a hedge subtype.

Unchanged from what you already have — reproduced here for continuity since they
sit between the layers you've built and the ones below.

In [60]:
def method_paper(resp, par, cf):
    return {"par_acc": int(contains(resp, par, sig_words(par))),
            "cf_acc":  int(contains(resp, cf,  sig_words(cf)))}

In [61]:
def method_ordered(resp, par, cf):
    pw, cw = discr_words(par, cf), discr_words(cf, par)
    ph, ch = contains(resp, par, pw), contains(resp, cf, cw)
    if not ph and not ch: return {"label":"OTHER","sub":"neither"}
    if ph and not ch:     return {"label":"PAR","sub":"par_only"}
    if ch and not ph:     return {"label":"CTX","sub":"ctx_only"}
    return ({"label":"CTX","sub":"ctx_then_par"}
            if first_pos(resp,cf,cw) < first_pos(resp,par,pw)
            else {"label":"PAR","sub":"par_then_cf"})

## Metrics — per-method R_ctx / R_par / R_other, bootstrap CIs

$$\text{mean} = \frac{1}{n}\sum_i \text{flag}_i, \qquad \text{CI}_{95\%} = [\hat q_{0.025}, \hat q_{0.975}]\ \text{over 1000 resamples with replacement}$$

For method $m$, over the filtered subset $S$ (items with `passed=True`):
$$R_\text{ctx}^{(m)} = \frac{1}{|S|}\sum_{i\in S}\mathbb{1}[\text{label}_i^{(m)}=\text{CTX}], \qquad R_\text{par}^{(m)},\ R_\text{other}^{(m)} \text{ analogously}$$

Computed for **every method separately** — `paper`, `ordered`, `logprob`, and
`final` (the actual pipeline output: Layer 1, backstopped by Layer 2 on AMBIG):

| Method | R_ctx | R_par | third bucket |
|---|---|---|---|
| `paper` | mean(cf_acc) | mean(par_acc) | — (cf_acc/par_acc can overlap; **not a partition**, don't compare its rates directly to the others) |
| `ordered` | CTX rate | PAR rate | `R_other` |
| `logprob` | CTX rate | PAR rate | `R_ambiguous` (unresolved, not OTHER) |
| `final` | CTX rate | PAR rate | `R_other` (this is the number that goes in the paper) |

Also logged per spec §5.5/§6: `escalation_rate` (share sent to the judge),
`layer2_failed_rate` (share of escalations where the API call itself failed —
tracked separately from genuine `OTHER`), and `other_breakdown` (hedged /
abstained / hallucinated, as a share of the full filtered subset — these three
plus `layer2_failed_rate` should sum to roughly `R_other`; a gap between them is
itself diagnostic of judge failures).

In [62]:
def rate_ci(flags, n_boot=1000, seed=0):
    n = len(flags)
    if n == 0: return {"mean": None, "ci": [None, None], "n": 0}
    rnd = random.Random(seed); mean = sum(flags)/n
    pts = sorted(sum(rnd.choice(flags) for _ in range(n))/n for _ in range(n_boot))
    return {"mean": round(mean,3), "ci": [round(pts[int(.025*n_boot)],3),
            round(pts[int(.975*n_boot)],3)], "n": n}

def _rates(subset, methods):
    d = {"n_passed": len(subset)}
    if not subset: return d

    if "paper" in methods:
        cf  = [r["paper"]["cf_acc"]==1  for r in subset]
        pa  = [r["paper"]["par_acc"]==1 for r in subset]
        d["paper"] = {"R_ctx": rate_ci(cf), "R_par": rate_ci(pa),
                      "note": "cf_acc/par_acc are independent containment checks, not a partition — can overlap or both be 0"}

    if "ordered" in methods:
        lbl = [r["ordered"]["label"] for r in subset]
        d["ordered"] = {"R_ctx": rate_ci([x=="CTX" for x in lbl]),
                         "R_par": rate_ci([x=="PAR" for x in lbl]),
                         "R_other": rate_ci([x=="OTHER" for x in lbl])}

    if "logprob" in methods:
        lbl = [r["logprob"]["label"] for r in subset]
        d["logprob"] = {"R_ctx": rate_ci([x=="CTX" for x in lbl]),
                         "R_par": rate_ci([x=="PAR" for x in lbl]),
                         "R_ambiguous": rate_ci([x=="AMBIG" for x in lbl])}

    finals = [r["final_label"] for r in subset if "final_label" in r]
    if finals:
        d["final"] = {"R_ctx": rate_ci([x=="CTX" for x in finals]),
                       "R_par": rate_ci([x=="PAR" for x in finals]),
                       "R_other": rate_ci([x=="OTHER" for x in finals])}

    escalated = [r for r in subset if r.get("logprob", {}).get("label") == "AMBIG"]
    d["escalation_rate"] = round(len(escalated) / len(subset), 3)
    failed = [r for r in escalated if r.get("judge", {}).get("layer2_failed")]
    d["layer2_failed_rate"] = round(len(failed) / len(subset), 3)

    judged_other = [r["judge"]["other_subtype"] for r in escalated
                    if r.get("judge", {}).get("label") == "OTHER" and r["judge"].get("other_subtype")]
    if judged_other:
        from collections import Counter
        cnt = Counter(judged_other)
        d["other_breakdown"] = {k: round(v/len(subset), 3) for k, v in cnt.items()}

    return d

def summarize(rows, methods):
    passed = [r for r in rows if r["passed"]]
    strata = sorted({r["source"] for r in rows})
    per_stratum = {}
    for s in strata:
        total_s = [r for r in rows if r["source"] == s]
        passed_s = [r for r in total_s if r["passed"]]
        stratum_rates = _rates(passed_s, methods)
        stratum_rates["filter_yield"] = round(len(passed_s) / max(len(total_s), 1), 3)
        per_stratum[s] = stratum_rates
    return {"filter_yield": round(len(passed)/max(len(rows),1),3),
            "aggregate":   _rates(passed, methods),
            "per_stratum": per_stratum}

## Unified evaluator — the manually-verifiable record

`evaluate()` runs Layer 0 (filter, from your Cell 8) → Layer 1 → Layer 2 (AMBIG
only) → Layer 3 (diagnostics) per item. With `verbose=True` (default), every
per-item record embeds enough to audit by eye without touching the dataset file:

- `question`, `context`, `parametric_answer`, `counterfactual_answer`
- `gen_prompt_text`, `score_prompt_text` — the **exact decoded prompts**, chat-template
  tokens included, so you can confirm the template applied correctly
- `response` — the actual generation
- `paper`, `ordered`, `logprob`, `judge` (if escalated) — every method's verdict
  and underlying numbers
- `final_label` — what actually counts toward $R_\text{ctx}$

Generation happens **once per item** and is reused for the judge call if
escalated — confirmed by an offline mock test (no duplicate `generate()` calls).
Set `verbose=False` for the real 20-checkpoint × 5-run SFT integration, once
you've validated correctness here, to avoid repeating static item text in every
checkpoint's JSON.

In [63]:
def evaluate(model, tok, items, use_ct, distractor_pool,
             methods=("paper","ordered","logprob"),
             record_gen_filter=False, tau=TAU, judge_fn_=None,
             checkpoint_id="adhoc", k_distractors=K_DISTRACTORS,
             verbose=True):
    need_gen = ("paper" in methods) or ("ordered" in methods) or (judge_fn_ is not None)
    rows = []
    for it in items:
        q, ctx = it["question"], it["context"]
        par, cf = it["parametric_answer"], it["counterfactual_answer"]
        rec = {"item_id": it["item_id"], "source": it["source"]}
        if verbose:
            rec.update({"question": q, "context": ctx,
                        "parametric_answer": par, "counterfactual_answer": cf})

        rec["passed"] = filter_logprob(model, tok, it, distractor_pool, use_ct, k=k_distractors)
        if record_gen_filter:
            rec["filter_gen"] = filter_generation(model, tok, q, par, use_ct)

        gen_ids   = build_gen_prompt_ids(tok, q, ctx, use_ct)
        score_ids = build_score_prompt_ids(tok, q, ctx, use_ct)
        if verbose:
            rec["gen_prompt_text"]   = tok.decode(gen_ids,   skip_special_tokens=False)
            rec["score_prompt_text"] = tok.decode(score_ids, skip_special_tokens=False)

        resp = None
        if need_gen:
            resp = generate(model, tok, gen_ids)
            rec["response"] = resp
            if "paper"   in methods: rec["paper"]   = method_paper(resp, par, cf)
            if "ordered" in methods: rec["ordered"] = method_ordered(resp, par, cf)

        if "logprob" in methods:
            rec["logprob"] = method_logprob(model, tok, score_ids, par, cf, tau)
            if rec["logprob"]["label"] == "AMBIG" and judge_fn_ is not None:
                if resp is None:
                    resp = generate(model, tok, gen_ids)
                    rec["response"] = resp
                rec["judge"] = judge_fn_(resp, q, ctx, par, cf,
                                          item_id=it["item_id"], checkpoint_id=checkpoint_id)
                rec["final_label"] = rec["judge"]["label"]
            else:
                rec["final_label"] = rec["logprob"]["label"]

        rows.append(rec)
    return {"per_item": rows, "summary": summarize(rows, methods)}

## Offline self-tests

Three things, all runnable with zero model calls:
1. Classifier regression (word-boundary fix) — unchanged from before.
2. Scorer index arithmetic (synthetic logits) — unchanged from before.
3. **New** — aggregation regression: confirms `_rates`/`summarize` compute
   `escalation_rate`, `layer2_failed_rate`, and `other_breakdown` correctly on
   synthetic rows with known answers, before trusting them on a real run.

In [64]:
CASES = [
    ("clean CTX",            "Chengdu.",                                 "Beijing","Chengdu","CTX"),
    ("clean PAR",             "Beijing.",                                 "Beijing","Chengdu","PAR"),
    ("substring bug: cf",     "The mississippian era rocks are old.",     "Amazon","Mississippi","OTHER"),
    ("substring bug: short",  "Usage statistics show growth.",            "USA","EU","OTHER"),
    ("prefix collision cf",   "Prizren.",                                 "Pristina","Prizren","CTX"),
    ("prefix collision par",  "Pristina.",                                "Pristina","Prizren","PAR"),
    ("hedge, ctx first",      "Chengdu, though in reality it's Beijing.", "Beijing","Chengdu","CTX"),
    ("abstention",            "I'm not sure I can answer that.",         "Beijing","Chengdu","OTHER"),
]
all_pass = True
for name, resp, par, cf, expected in CASES:
    got = method_ordered(resp, par, cf)["label"]
    ok = got == expected
    all_pass &= ok
    print(f"{name:22s} {got:7s} {'PASS' if ok else f'FAIL (expected {expected})'}")
assert all_pass
print("\nALL CLASSIFIER TESTS PASS")

clean CTX              CTX     PASS
clean PAR              PAR     PASS
substring bug: cf      OTHER   PASS
substring bug: short   OTHER   PASS
prefix collision cf    CTX     PASS
prefix collision par   PAR     PASS
hedge, ctx first       CTX     PASS
abstention             OTHER   PASS

ALL CLASSIFIER TESTS PASS


In [65]:
def _test_scorer_indexing():
    V, seq, prompt_len = 50, [5,9,2,7,40,41,42], 4
    ans = torch.tensor(seq[prompt_len:])
    logits = torch.full((len(seq), V), -30.0)
    for i in range(len(seq)-1):
        logits[i, seq[i+1]] = 30.0
    n = prompt_len
    lp = torch.log_softmax(logits[n-1:-1, :].float(), dim=-1)
    tok_lp = lp[torch.arange(ans.shape[0]), ans]
    assert torch.allclose(tok_lp, torch.zeros_like(tok_lp), atol=1e-4), tok_lp
    assert logits[n-1:-1,:].shape[0] == ans.shape[0]
    print("scorer indexing test: PASS")
_test_scorer_indexing()

scorer indexing test: PASS


In [66]:
def _test_aggregation():
    rows = [
        {"item_id":"a1","source":"country_capitals","passed":True,
         "paper":{"cf_acc":1,"par_acc":0}, "ordered":{"label":"CTX","sub":"ctx_only"},
         "logprob":{"label":"CTX","delta":1.5}, "final_label":"CTX"},
        {"item_id":"a2","source":"country_capitals","passed":True,
         "paper":{"cf_acc":0,"par_acc":1}, "ordered":{"label":"PAR","sub":"par_only"},
         "logprob":{"label":"PAR","delta":-2.1}, "final_label":"PAR"},
        {"item_id":"a3","source":"country_capitals","passed":True,
         "paper":{"cf_acc":1,"par_acc":1}, "ordered":{"label":"CTX","sub":"ctx_then_par"},
         "logprob":{"label":"AMBIG","delta":0.3},
         "judge":{"label":"OTHER","other_subtype":"hedged","layer2_failed":False}, "final_label":"OTHER"},
        {"item_id":"a4","source":"world_facts","passed":False,
         "paper":{"cf_acc":0,"par_acc":0}, "ordered":{"label":"OTHER","sub":"neither"},
         "logprob":{"label":"AMBIG","delta":0.1},
         "judge":{"label":"CTX","other_subtype":None,"layer2_failed":False}, "final_label":"CTX"},
        {"item_id":"a5","source":"world_facts","passed":True,
         "paper":{"cf_acc":0,"par_acc":0}, "ordered":{"label":"OTHER","sub":"neither"},
         "logprob":{"label":"AMBIG","delta":-0.05},
         "judge":{"label":"OTHER","other_subtype":None,"layer2_failed":True}, "final_label":"OTHER"},
    ]
    res = summarize(rows, methods=("paper","ordered","logprob"))
    a = res["aggregate"]
    assert a["n_passed"] == 4
    assert a["final"]["R_ctx"]["mean"] == 0.25 and a["final"]["R_par"]["mean"] == 0.25 and a["final"]["R_other"]["mean"] == 0.5
    assert a["escalation_rate"] == 0.5 and a["layer2_failed_rate"] == 0.25
    assert a["other_breakdown"] == {"hedged": 0.25}
    assert res["per_stratum"]["country_capitals"]["n_passed"] == 3
    assert res["per_stratum"]["world_facts"]["n_passed"] == 1
    print("aggregation test: PASS")
_test_aggregation()

aggregation test: PASS


## Manual preflight — real adversarial items, judge included

Same six items as before (prefix collision, shared-word collision, multi-word
biography swaps, a trap capital, a genuinely confusable pair). Now also shows the
judge's verdict (and `other_subtype`) whenever Layer 1 is AMBIG, and the decoded
score/gen prompts, so a disagreement can be traced back to the exact text the
model saw.

In [79]:
MANUAL_ITEM_IDS = ["cap_0091", "wf_0039", "bio_0000", "cap_0008", "wf_0023", "bio_0014"]
MANUAL_ITEMS = [id_map[i] for i in MANUAL_ITEM_IDS]

def _disagree(pm, om, lp):
    p = 'CTX' if pm["cf_acc"] == 1 else 'x'
    o = 'CTX' if om["label"] == "CTX" else 'x'
    if lp["label"] == "AMBIG": return p != o
    return len({p, o, 'CTX' if lp["label"] == "CTX" else 'x'}) > 1

def preflight(model, tok, manual_items, pool, use_ct, tau=TAU, judge_fn_=None, checkpoint_id="preflight"):
    print(f"PRE-FLIGHT  {len(manual_items)} item(s)\n")
    for it in manual_items:
        q, ctx = it["question"], it["context"]
        par, cf = it["parametric_answer"], it["counterfactual_answer"]
        print(f"  {it['item_id']}  par={par!r}  cf={cf!r}")
        for name, a in [("PAR", par), ("CF", cf)]:
            ta = tok(" " + a.strip(), add_special_tokens=False).input_ids
            print(f"    tokens {name}: {[tok.decode([t]) for t in ta]}")

        gen_ids_noctx = build_gen_prompt_ids(tok, q, None, use_ct)
        gen_ids       = build_gen_prompt_ids(tok, q, ctx, use_ct)
        score_ids     = build_score_prompt_ids(tok, q, ctx, use_ct)

        no_ctx = generate(model, tok, gen_ids_noctx)
        resp   = generate(model, tok, gen_ids)
        passed = filter_logprob(model, tok, it, pool, use_ct)
        pm     = method_paper(resp, par, cf)
        om     = method_ordered(resp, par, cf)
        lp     = method_logprob(model, tok, score_ids, par, cf, tau)

        final = lp["label"]
        judge_note = ""
        if lp["label"] == "AMBIG" and judge_fn_ is not None:
            j = judge_fn_(resp, q, ctx, par, cf, item_id=it["item_id"], checkpoint_id=checkpoint_id)
            final = j["label"]
            sub = f" ({j['other_subtype']})" if j.get("other_subtype") else ""
            judge_note = f"  [judge -> {j['label']}{sub}]"

        print(f"    no-ctx  : {no_ctx[:70]!r}  filter_pass={passed}")
        print(f"    with-ctx: {resp[:80]!r}")
        print(f"    paper   : cf_acc={pm['cf_acc']}  par_acc={pm['par_acc']}")
        print(f"    ordered : {om['label']} ({om['sub']})")
        print(f"    logprob : delta={lp['delta']:+.3f}  verdict={lp['label']}  par_pt={lp['lp_par_pt']}  cf_pt={lp['lp_cf_pt']}{judge_note}")
        print(f"    FINAL   : {final}")
        print(f"    status  : {'DISAGREE — inspect' if _disagree(pm, om, lp) else 'agree'}\n")

distractor_pool = build_distractor_pool(items)   # if not already built in your Cell 8

_m, _t = load_model(MODELS[0]["id"])
preflight(_m, _t, MANUAL_ITEMS, distractor_pool, MODELS[0]["use_chat_template"], judge_fn_=judge_fn)
free_model(_m)

Loading meta-llama/Llama-3.1-8B-Instruct  precision=bf16


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Memory footprint: 16.06 GB (precision=bf16)
PRE-FLIGHT  6 item(s)

  cap_0091  par='Pristina'  cf='Prizren'
    tokens PAR: [' Pr', 'ist', 'ina']
    tokens CF: [' Pr', 'iz', 'ren']
    no-ctx  : 'Pristina.'  filter_pass=True
    with-ctx: 'Pristina.'
    paper   : cf_acc=0  par_acc=1
    ordered : PAR (par_only)
    logprob : delta=-0.083  verdict=AMBIG  par_pt=-0.241  cf_pt=-0.324  [judge -> PAR]
    FINAL   : PAR
    status  : agree

  wf_0039  par='United States'  cf='United Kingdom'
    tokens PAR: [' United', ' States']
    tokens CF: [' United', ' Kingdom']
    no-ctx  : 'Panama.'  filter_pass=True
    with-ctx: 'The United Kingdom.'
    paper   : cf_acc=1  par_acc=1
    ordered : CTX (ctx_only)
    logprob : delta=+1.125  verdict=CTX  par_pt=-1.715  cf_pt=-0.59
    FINAL   : CTX
    status  : agree

  bio_0000  par='Leonardo da Vinci'  cf='Enrico Fermi'
    tokens PAR: [' Leonardo', ' da', ' Vinci']
    tokens CF: [' En', 'rico', ' Ferm', 'i']
    no-ctx  : 'Leonardo da Vinci

In [68]:
def print_summary(model_id, res):
    s = res["summary"]; print(f"\n### {model_id}  | filter_yield={s['filter_yield']}")
    a = s["aggregate"]
    if "final" in a: print(f"  FINAL   R_ctx={a['final']['R_ctx']['mean']}  R_par={a['final']['R_par']['mean']}  R_other={a['final']['R_other']['mean']}")

In [69]:
def run_sanity_check(base_id="meta-llama/Llama-3.1-8B", instruct_id="meta-llama/Llama-3.1-8B-Instruct"):
    results = {}
    for label, model_id, use_ct in [("base", base_id, False), ("instruct", instruct_id, True)]:
        model, tok = load_model(model_id)
        res = evaluate(model, tok, items, use_ct, distractor_pool,
                        methods=("paper","ordered","logprob"), record_gen_filter=True,
                        tau=TAU, judge_fn_=judge_fn, checkpoint_id=f"sanity_{label}", verbose=True)
        results[label] = res
        print_summary(model_id, res)
        json.dump(res, open(f"{OUT_DIR}/sanity_{label}.json", "w"), indent=2)
        free_model(model)

    r_base = results["base"]["summary"]["aggregate"]["final"]["R_ctx"]["mean"]
    r_instruct = results["instruct"]["summary"]["aggregate"]["final"]["R_ctx"]["mean"]
    fy_base = results["base"]["summary"]["filter_yield"]

    print(f"\nSANITY CHECK: R_ctx(base)={r_base}  R_ctx(instruct)={r_instruct}  filter_yield(base)={fy_base}")
    if not (0.40 <= r_base <= 0.70):
        print(f"  WARNING: R_ctx(base)={r_base} outside spec's expected 40-70% — check prompt formatting.")
    if fy_base < 0.80:
        print(f"  WARNING: filter_yield(base)={fy_base} below spec's expected >=80% (capitals/world_facts).")
    if abs(r_base - r_instruct) < 0.10:
        print(f"  WARNING: R_ctx(base) and R_ctx(instruct) within 10 points — spec expects instruct "
              f"substantially lower. Scoring may be broken.")
    else:
        print(f"  R_ctx dropped {r_base - r_instruct:+.3f} base->instruct — consistent with post-inversion state.")
    return results

In [70]:
def paired_ctx_flags_from_checkpoints(rows_a, rows_b):
    """Persistently-filtered subset per spec §6.1: items passing the filter at BOTH checkpoints."""
    a_map = {r["item_id"]: r for r in rows_a if r.get("passed")}
    b_map = {r["item_id"]: r for r in rows_b if r.get("passed")}
    common_ids = sorted(set(a_map) & set(b_map))
    pairs = [(a_map[i]["final_label"] == "CTX", b_map[i]["final_label"] == "CTX") for i in common_ids]
    return pairs, len(common_ids)

def mcnemar_test(paired_ctx_flags):
    """Exact two-sided McNemar (binomial on discordant pairs) -- appropriate for the
    typically-small discordant counts here, rather than the chi-square approximation."""
    n01 = sum(1 for a, b in paired_ctx_flags if (not a) and b)   # switched TO ctx
    n10 = sum(1 for a, b in paired_ctx_flags if a and (not b))   # switched FROM ctx
    n = n01 + n10
    if n == 0:
        return {"n01": 0, "n10": 0, "n_discordant": 0, "p_value": 1.0}
    k = min(n01, n10)
    def binom_cdf(k, n, p=0.5):
        return sum(math.comb(n, i) * (p**i) * ((1-p)**(n-i)) for i in range(k+1))
    p_value = min(1.0, 2 * binom_cdf(k, n))
    return {"n01": n01, "n10": n10, "n_discordant": n, "p_value": round(p_value, 4)}

## Experiment3: Method Agreement Test

NameError: name 'results' is not defined